# VisiPruner trên SmolVLM · Deep Discard toàn bộ test

Chạy baseline và bảy lượt **Discard từ từng layer 24–30 trở đi** trên **16.817 câu DriveLM test**. Layer đánh số từ 1. Cả 8 cấu hình dùng cùng checkpoint và greedy decoding với trần 512 token.

Bật Internet và GPU. Trên T4 ×2, cả bốn cặp chạy hai cấu hình song song. Trên một GPU, mỗi cặp chạy tuần tự. Log tiến độ lưu theo cấu hình trong `logs/`.


In [ ]:
%%writefile /kaggle/working/visipruner-deep-full.env
REPOSITORY=https://github.com/huyvanzzz/Log_Metrics.git
SOURCE_BRANCH=smolvlm-attention-nguyet
SOURCE_DIR=/kaggle/working/log-metrics-visipruner
DATA_DIR=/kaggle/working/drivelm-nuscenes-v1-0
export HF_TOKEN_SECRET=''
OUTPUT_DIR=/kaggle/working/visipruner_deep_discard_512_outputs
RESUME_FROM=''
BATCH_SIZE=8
MAX_NEW_TOKENS=512


## Cài đặt

Cập nhật mã nguồn và cài thư viện chạy mô hình, chấm điểm và vẽ hình.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
if [ -d "$SOURCE_DIR/.git" ]; then
  test "$(git -C "$SOURCE_DIR" branch --show-current)" = "$SOURCE_BRANCH"
  test -z "$(git -C "$SOURCE_DIR" status --porcelain --untracked-files=no)"
  git -C "$SOURCE_DIR" pull --ff-only origin "$SOURCE_BRANCH"
else
  git clone --depth 1 --single-branch --branch "$SOURCE_BRANCH" "$REPOSITORY" "$SOURCE_DIR"
fi
python -m pip install -q -r "$SOURCE_DIR/smolvlm_attention/requirements.txt" pycocoevalcap==1.2
if ! command -v java >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq default-jre-headless
fi


## Khôi phục phiên trước (tùy chọn)

Nếu chia qua nhiều phiên, lưu thư mục đầu ra, gắn bản lưu làm Kaggle Input và đặt `RESUME_FROM` thành thư mục chứa `manifest.json`. Cell dưới sao chép vào thư mục đầu ra mới. Với đầu ra đã có trong phiên hiện tại, chạy lại các cell đo để tiếp tục từ bộ ảnh cuối đã lưu.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
if [ -n "$RESUME_FROM" ] && [ ! -e "$OUTPUT_DIR" ]; then
  test -f "$RESUME_FROM/manifest.json"
  mkdir -p "$(dirname "$OUTPUT_DIR")"
  staging=$(mktemp -d "${OUTPUT_DIR}.restore.XXXXXX")
  trap 'rm -rf -- "$staging"' EXIT
  cp -R -- "$RESUME_FROM"/. "$staging/"
  chmod -R u+rwX -- "$staging"
  mv -T -- "$staging" "$OUTPUT_DIR"
  trap - EXIT
fi


## Chuẩn bị dữ liệu

Dùng toàn bộ split test, giữ nguyên thứ tự và ID câu hỏi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.mvpruner_ablation fetch-hf --output "$DATA_DIR"
python -u -m smolvlm_attention.visipruner_ablation prepare --suite deep-full --test "$DATA_DIR/multi_frame/multi_frame_test.json" --image-root "$DATA_DIR" --output "$OUTPUT_DIR"


## 01–02/08 · Baseline và Discard 24

- **Baseline** giữ đủ token ảnh và kiểm tra khớp với decoder gốc.
- **Discard 24** bỏ token ảnh từ layer 24 đến hết. Cả hai dùng chung cache đặc trưng ảnh.


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants baseline deep-discard-24 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 03–04/08 · Discard 25 và 26

Mỗi cấu hình loại token ảnh khỏi chuỗi và KV cache từ mốc riêng của nó đến hết.


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-discard-25 deep-discard-26 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 05–06/08 · Discard 27 và 28

Mỗi cấu hình loại token ảnh khỏi chuỗi và KV cache từ mốc riêng của nó đến hết.


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-discard-27 deep-discard-28 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 07–08/08 · Discard 29 và 30

Mỗi cấu hình loại token ảnh khỏi chuỗi và KV cache từ mốc riêng của nó đến hết.


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-discard-29 deep-discard-30 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Chấm điểm và vẽ hình

Xuất bảng điểm và hai hình PNG/PDF cho ΔOverall và ΔYes/No accuracy so với baseline toàn tập.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation score --output "$OUTPUT_DIR"


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation plot --output "$OUTPUT_DIR"
